# Sprint 2 — Data Pipeline: SQL → Cleaning → Preprocessing

**Project:** Speech Emotion Recognition (SER)  
**Sprint:** 2  

## What this notebook does

This notebook covers the complete data pipeline from raw audio files to model-ready `tf.data` batches:

1. **SQL integration** — connect to the Aiven MySQL database, populate the `clips` metadata table with file paths, emotion labels, and train/test split assignments, then query each split back out.
2. **Data cleaning** — filter to the two highest-quality sources (CREMA-D and RAVDESS), remove the mislabelled *Surprised* class, and enforce a 1–3.5 s duration window.
3. **Preprocessing pipeline** — extract a 4-channel acoustic feature tensor (perceptual spectrogram, ZCR, spectral centroid, RMS energy) for each clip and wrap everything in a `tf.data.Dataset`.
4. **Verification** — confirm that a sample batch has the expected shape, dtype, and value range before handing off to the model.

> **Split note:** This notebook uses the same **80/20 stratified random split** (`random_state=42`) as `MelCNN.ipynb`, ensuring both notebooks operate on identical train and test sets.

**Reproducibility:** Run all cells top-to-bottom in a fresh kernel. The only external requirement is a populated `.env` file (or equivalent Colab/Kaggle secrets) with the Aiven MySQL credentials.

---
## 0 · Environment Setup

In [23]:
import sys, os, re, glob
from pathlib import Path

import numpy as np
import pandas as pd

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB  = "google.colab" in sys.modules
if not ON_COLAB:
    try:
        import google.colab
        ON_COLAB = True
    except ImportError:
        pass

if ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working")
    ENV     = "Kaggle"
elif ON_COLAB:
    OUT_DIR = Path("/content")
    ENV     = "Colab"
else:
    OUT_DIR = Path(".")
    ENV     = "Local"

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=True)
except ImportError:
    pass

import tensorflow as tf
print(f"Environment : {ENV}")
print(f"Output dir  : {OUT_DIR}")
print(f"TF version  : {tf.__version__}")
print(f"GPUs        : {tf.config.list_physical_devices('GPU')}")

Environment : Local
Output dir  : .
TF version  : 2.21.0
GPUs        : []


---
## 1 · SQL Connection

We use **Aiven MySQL** as the central metadata store. Credentials are loaded from environment variables so no secrets are hard-coded in this notebook.

The `clips` table schema:

| Column | Type | Description |
|---|---|---|
| `clip_id` | INT (PK) | Auto-assigned row ID |
| `filename` | VARCHAR(255) | Original audio filename |
| `source` | VARCHAR(16) | Dataset origin (CREMA-D / RAVDESS) |
| `emotion` | VARCHAR(16) | Ground-truth emotion label |
| `duration` | FLOAT | Clip length in seconds |
| `path` | VARCHAR(512) | Absolute path on the current machine |
| `split` | VARCHAR(8) | train / test |

In [24]:
from sqlalchemy import create_engine, text, types

MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PWD  = os.getenv("MYSQL_PASSWORD", "")
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = os.getenv("MYSQL_PORT", "3306")
MYSQL_DB   = os.getenv("MYSQL_DB",   "defaultdb")
SSL_CA     = os.getenv("MYSQL_SSL_CA", "")

connect_args = {"ssl": {"ca": SSL_CA}} if SSL_CA else {"ssl": {"ssl": True}}

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PWD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}",
    connect_args=connect_args,
    pool_pre_ping=True,
)

with engine.connect() as conn:
    version = conn.execute(text("SELECT VERSION()")).scalar()
print(f"Connected to MySQL {version} at {MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}")

Connected to MySQL 8.4.8 at capstoneteam1-capstoneteam1.l.aivencloud.com:12383/defaultdb


---
## 2 · Data Cleaning

### 2.1 Locate the dataset

In [25]:
KNOWN_EMOTIONS = {"Angry", "Disgusted", "Fearful", "Happy", "Neutral", "Sad"}

def find_emotions_dir(root):
    for c in glob.glob(os.path.join(str(root), "**", ""), recursive=True):
        try:
            subs = {d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))}
        except OSError:
            continue
        if len(KNOWN_EMOTIONS & subs) >= 3:
            return Path(c)
    return None

CANDIDATES = [
    Path("/kaggle/input"),
    Path("/content/Emotions"),
    Path("Emotions"),
    Path("../Emotions"),
    Path.home() / "Downloads" / "Emotions",
]
EMO_DIR = None
for candidate in CANDIDATES:
    if candidate.exists():
        found = find_emotions_dir(candidate)
        if found:
            EMO_DIR = found
            break

if EMO_DIR is None and ON_COLAB:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("uldisvalainis/audio-emotions")
    EMO_DIR = find_emotions_dir(DATA_ROOT)

assert EMO_DIR is not None, (
    "Emotions folder not found.\n"
    "Kaggle: attach uldisvalainis/audio-emotions via Add Input.\n"
    "Colab : upload Emotions/ to /content/ or configure kaggle.json."
)
print("Dataset root:", EMO_DIR.resolve())

Dataset root: /Users/sa08/Group-1---Capstone/Emotions


### 2.2 Build the raw inventory and apply filters

In [26]:
import librosa
from tqdm.auto import tqdm

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rows.append({"filename": wav.name, "emotion": emo_path.name,
                     "source": source_of(wav.name), "path": str(wav)})
df_raw = pd.DataFrame(rows)
print(f"Raw inventory : {len(df_raw)} clips across {df_raw['source'].nunique()} sources")
df_raw["source"].value_counts()

Raw inventory : 12798 clips across 4 sources


source
CREMA-D    7442
TESS       2800
RAVDESS    2076
SAVEE       480
Name: count, dtype: int64

In [27]:
# Decision 1 — keep CREMA-D + RAVDESS only
df = df_raw[df_raw["source"].isin(["CREMA-D", "RAVDESS"])].copy()
print(f"After source filter : {len(df)} clips")

# Decision 2 — drop Surprised
df = df[df["emotion"] != "Suprised"].reset_index(drop=True)
print(f"After emotion filter: {len(df)} clips, {df['emotion'].nunique()} emotions")

# Decision 3 — duration window 1.0 – 3.5 s
df["duration"] = [
    round(librosa.get_duration(path=p), 3)
    for p in tqdm(df["path"], desc="duration")
]
before = len(df)
df = df[(df["duration"] >= 1.0) & (df["duration"] <= 3.5)].reset_index(drop=True)
print(f"After duration filter: removed {before - len(df)} clips → {len(df)} remain")
print("\nEmotion distribution:")
print(df["emotion"].value_counts())

After source filter : 9518 clips
After emotion filter: 9326 clips, 6 emotions


duration: 100%|██████████| 9326/9326 [00:00<00:00, 21602.40it/s]

After duration filter: removed 1983 clips → 7343 remain

Emotion distribution:
emotion
Happy        1313
Fearful      1307
Sad          1239
Angry        1236
Disgusted    1136
Neutral      1112
Name: count, dtype: int64


### 2.3 Assign train / test splits

**Decision 4 — 80/20 stratified random split**  
We use a stratified random split so each emotion class is proportionally represented in both sets. `random_state=42` ensures the split is identical to the one used in `MelCNN.ipynb`, making results directly comparable across notebooks.

In [28]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df["emotion"]
)

df_train = df_train.copy(); df_train["split"] = "train"
df_test  = df_test.copy();  df_test["split"]  = "test"

df_all = pd.concat([df_train, df_test]).reset_index(drop=True)
df_all.insert(0, "clip_id", range(1, len(df_all) + 1))

print(f"train : {len(df_train)} clips")
print(f"test  : {len(df_test)} clips")
print("\nEmotion distribution per split:")
print(df_all.groupby(["split", "emotion"]).size().unstack(fill_value=0))

train : 5874 clips
test  : 1469 clips

Emotion distribution per split:
emotion  Angry  Disgusted  Fearful  Happy  Neutral  Sad
split                                                  
test       247        227      262    263      222  248
train      989        909     1045   1050      890  991


---
## 3 · Write Metadata to SQL

Push the cleaned, split-assigned DataFrame to the `clips` table. `if_exists="replace"` makes this cell idempotent — re-running always produces a clean, consistent table.

In [29]:
TABLE = "clips"

cols = ["clip_id", "filename", "source", "emotion", "duration", "path", "split"]

with engine.begin() as conn:
    # Drop and recreate with PK defined upfront (Aiven requires PK at creation time)
    conn.execute(text(f"DROP TABLE IF EXISTS {TABLE}"))
    conn.execute(text(f"""
        CREATE TABLE {TABLE} (
            clip_id  INT          NOT NULL,
            filename VARCHAR(255),
            source   VARCHAR(16),
            emotion  VARCHAR(16),
            duration FLOAT,
            path     VARCHAR(512),
            split    VARCHAR(8),
            PRIMARY KEY (clip_id),
            INDEX idx_emotion (emotion),
            INDEX idx_split   (split)
        )
    """))
    print(f"Table `{TABLE}` created")

# Append data into the already-created table
df_all[cols].to_sql(TABLE, engine, if_exists="append", index=False)

with engine.connect() as conn:
    n = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE}")).scalar()

print(f"Wrote {n} rows to `{MYSQL_DB}.{TABLE}`")

Table `clips` created
Wrote 7343 rows to `defaultdb.clips`


---
## 4 · Query Splits from SQL

All downstream steps read from the database rather than the in-memory DataFrame, validating the round-trip.

In [30]:
with engine.connect() as conn:
    dftrain = pd.read_sql(text("SELECT * FROM clips WHERE split = 'train'"), conn)
    dftest  = pd.read_sql(text("SELECT * FROM clips WHERE split = 'test'"),  conn)

print(f"train : {len(dftrain)} clips")
print(f"test  : {len(dftest)} clips")
dftrain.head(3)

train : 5874 clips
test  : 1469 clips


,clip_id,filename,source,emotion,duration,path,split
0,1,1060_IOM_DIS_XX.wav,CREMA-D,Disgusted,2.636,../Emotions/Disgusted/1060_IOM_DIS_XX.wav,train
1,2,1085_TAI_ANG_XX.wav,CREMA-D,Angry,2.636,../Emotions/Angry/1085_TAI_ANG_XX.wav,train
2,3,1004_ITH_ANG_XX.wav,CREMA-D,Angry,3.270,../Emotions/Angry/1004_ITH_ANG_XX.wav,train


---
## 5 · Preprocessing Pipeline

### 5.1 Feature extraction

Each audio clip is converted into a **4-channel 2D tensor** of shape `(257, 110, 4)`:

| Channel | Feature | Why |
|---|---|---|
| 0 | A-weighted (perceptual) log power spectrogram | Captures timbre weighted to human hearing |
| 1 | Zero-crossing rate (tiled) | Noisiness and frication — discriminates Angry/Fearful |
| 2 | Spectral centroid (tiled) | Brightness — correlates with valence |
| 3 | RMS energy (tiled) | Loudness / arousal |

**Parameters:** SR = 16 000 Hz · N_FFT = 512 · HOP = 512 · MAX_FRAMES = 110 (≈ 3.5 s)

In [31]:
import gc

SR         = 16000
N_FFT      = 512
HOP        = 512
N_BINS     = N_FFT // 2 + 1   # 257
MAX_FRAMES = 110              # 110 × 512 / 16000 ≈ 3.5 s
FREQS      = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)

def _fit2d(m):
    if m.shape[1] < MAX_FRAMES:
        return np.pad(m, ((0, 0), (0, MAX_FRAMES - m.shape[1])))
    return m[:, :MAX_FRAMES]

def _fit1d(a):
    a = a[:MAX_FRAMES] if len(a) >= MAX_FRAMES else np.pad(a, (0, MAX_FRAMES - len(a)))
    return np.tile(a, (N_BINS, 1))

def extract_features(path):
    y, sr = librosa.load(path, sr=SR)
    S    = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2
    perc = librosa.perceptual_weighting(S, FREQS)
    zcr  = librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0]
    cen  = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP)[0]
    rms  = librosa.feature.rms(y=y, hop_length=HOP)[0]
    return np.stack([_fit2d(perc), _fit1d(zcr), _fit1d(cen), _fit1d(rms)], axis=-1)

print(f"Output shape per clip: ({N_BINS}, {MAX_FRAMES}, 4)")

Output shape per clip: (257, 110, 4)


### 5.2 Extract features for train and test

Features are pre-allocated (not built as a Python list) to halve peak RAM usage. A `gc.collect()` runs every 500 clips to release librosa's internal buffers. Results are cached so re-running the notebook skips re-extraction.

In [32]:
def build_features(df_split, desc):
    paths = list(df_split["path"])
    X = np.empty((len(paths), N_BINS, MAX_FRAMES, 4), dtype="float32")
    for i, p in enumerate(tqdm(paths, desc=desc)):
        X[i] = extract_features(p)
        if i % 500 == 0 and i > 0:
            gc.collect()
    return X

CACHE = OUT_DIR / "sprint2_features.npz"
if CACHE.exists():
    dat     = np.load(str(CACHE))
    X_train = dat["X_train"]
    X_test  = dat["X_test"]
    print("Loaded from cache:", X_train.shape)
else:
    X_train = build_features(dftrain, "train")
    X_test  = build_features(dftest,  "test")
    np.savez_compressed(str(CACHE), X_train=X_train, X_test=X_test)
    print("Extracted + cached")

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

train:   0%|          | 0/5874 [00:00<?, ?it/s]/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/librosa/core/convert.py:1869: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)
train:   0%|          | 1/5874 [00:00<20:49,  4.70it/s]/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/librosa/core/convert.py:1869: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)
test: 100%|██████████| 1469/1469 [00:03<00:00, 391.38it/s]


Extracted + cached
X_train : (5874, 257, 110, 4)
X_test  : (1469, 257, 110, 4)


### 5.3 Normalisation

Each of the 4 feature channels occupies a different numerical range (dB, ratios, Hz). We apply **per-channel z-score normalisation** using statistics computed on the training set only — the test set uses the same train mean/std to prevent data leakage.

In [33]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Per-channel normalisation — stats from train only
axes  = (0, 1, 2)
mean  = X_train.mean(axis=axes, keepdims=True)
std   = X_train.std(axis=axes,  keepdims=True)

X_train_n = (X_train - mean) / (std + 1e-9)
X_test_n  = (X_test  - mean) / (std + 1e-9)

# Encode labels
le = LabelEncoder()
y_train = to_categorical(le.fit_transform(dftrain["emotion"]))
y_test  = to_categorical(le.transform(dftest["emotion"]))

print("Classes:", list(le.classes_))
print(f"X_train_n : {X_train_n.shape}  y_train : {y_train.shape}")
print(f"X_test_n  : {X_test_n.shape}  y_test  : {y_test.shape}")

Classes: ['Angry', 'Disgusted', 'Fearful', 'Happy', 'Neutral', 'Sad']
X_train_n : (5874, 257, 110, 4)  y_train : (5874, 6)
X_test_n  : (1469, 257, 110, 4)  y_test  : (1469, 6)


### 5.4 tf.data.Dataset setup

Each split is wrapped in a `tf.data.Dataset` with shuffle (train only), batching, and prefetch to overlap data loading with GPU compute.

In [34]:
BATCH_TRAIN = 32
BATCH_TEST  = 64
AUTOTUNE    = tf.data.AUTOTUNE

def make_dataset(X, y, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype("float32"), y.astype("float32")))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train_n, y_train, BATCH_TRAIN, shuffle=True)
test_ds  = make_dataset(X_test_n,  y_test,  BATCH_TEST)

print("train_ds:", train_ds)
print("test_ds :", test_ds)

train_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 257, 110, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None, 6), dtype=tf.float32, name=None))>
test_ds : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 257, 110, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None, 6), dtype=tf.float32, name=None))>


---
## 6 · Verification

Pull one batch from each dataset and confirm shape, dtype, value range, and label validity.

In [35]:
for name, ds in [("train", train_ds), ("test", test_ds)]:
    X_batch, y_batch = next(iter(ds))
    x = X_batch.numpy()
    y = y_batch.numpy()
    print(f"\n── {name} batch ──")
    print(f"  X shape   : {x.shape}")
    print(f"  y shape   : {y.shape}")
    print(f"  X dtype   : {x.dtype}")
    print(f"  y dtype   : {y.dtype}")
    print(f"  X range   : [{x.min():.3f}, {x.max():.3f}]")
    print(f"  X has NaN : {np.isnan(x).any()}")
    print(f"  X has inf : {np.isinf(x).any()}")
    print(f"  y sum==1  : {np.allclose(y.sum(axis=1), 1.0)}")


── train batch ──
  X shape   : (32, 257, 110, 4)
  y shape   : (32, 6)
  X dtype   : float32
  y dtype   : float32
  X range   : [-6.315, 10.040]
  X has NaN : False
  X has inf : False
  y sum==1  : True

── test batch ──
  X shape   : (64, 257, 110, 4)
  y shape   : (64, 6)
  X dtype   : float32
  y dtype   : float32
  X range   : [-7.205, 11.628]
  X has NaN : False
  X has inf : False
  y sum==1  : True


---
## 7 · Summary

### What was done

1. **Connected to Aiven MySQL** and created a `clips` table storing metadata for every audio clip, with a `split` column assigning each to train or test.
2. **Cleaned the raw dataset** with three filters: source (CREMA-D + RAVDESS only), label (dropped misspelled *Suprised*), and duration (1.0–3.5 s window).
3. **Split the data** 80/20 (train/test) using stratified random sampling — identical to `MelCNN.ipynb` (`random_state=42`).
4. **Extracted a 4-channel acoustic feature tensor** (257 × 110 × 4) per clip: perceptual spectrogram, ZCR, spectral centroid, RMS energy at SR = 16 kHz.
5. **Normalised** each channel using training-set statistics only (no leakage into test).
6. **Wrapped each split** in a `tf.data.Dataset` with shuffling, batching, and prefetching.
7. **Verified** one batch from each split — shapes, dtypes, value ranges, and label validity all pass.

### Key decisions

| Decision | Rationale |
|---|---|
| CREMA-D + RAVDESS only | Most diverse, studio-quality, matched emotion sets |
| Drop Surprised | Absent from CREMA-D; misspelled folder; source-confounded label |
| 1.0–3.5 s window | Minimum content floor; max matches the fixed-length feature tensor |
| 80/20 random split | Matches MelCNN.ipynb exactly; stratified to keep class balance |
| Per-channel z-score | Features span different ranges — channel-wise scaling normalises each independently |

### Ready for Sprint 3

The `train_ds` and `test_ds` objects produced by this notebook are ready to pass directly into a Keras `model.fit()` call. The normalisation statistics (`mean`, `std`) are saved alongside the feature cache for use in inference.